# command-line entry point

## 1. purpose :
#### CLI =Command-Line Interface.

A CLI project is a normal Python project where users interact with your functions through terminal commands.

Example: 

Without a CLI, you might have to open Python and write:
``` python
cmd_explore(config)
```
With a CLI, we can simply type:

``` bash
python -m medimageforge info
python -m medimageforge explore
python -m medimageforge inspect 049
```

## 2. how make CLI 
1. Start with a normal Python function
``` python
def cmd_hello():
    print("Hello from MedImageForge!")
``` 
2. Connect the function to argparse

You create a CLI entry point:

``` python 
import argparse

def cmd_hello():
    print("Hello from MedImageForge!")


def main():
    parser = argparse.ArgumentParser()

    sub = parser.add_subparsers(dest="command", required=True)

    sub.add_parser("hello")

    args = parser.parse_args()

    if args.command == "hello":
        return cmd_hello()


if __name__ == "__main__":
    main()
```

Now the terminal understands:
``` python 
python app.py hello
```





## 2. The overall architecture
```
                    Terminal
                       │
                       ▼
          python -m medimageforge
                       │
                       ▼
                    main()
                       │
             ┌─────────┼─────────┐
             ▼         ▼         ▼
           info     explore    inspect
             │         │         │
             ▼         ▼         ▼
           config    dataset    images
                     checks     + masks
```


## 3. Functions in this file

| Function        | What it does                                                                                                 | Why we have it                                                                       |
| --------------- | ------------------------------------------------------------------------------------------------------------ | ------------------------------------------------------------------------------------ |
| `cmd_info()`    | Shows the project version, logging level, and whether configured paths/files exist                           | **Smoke test** — confirms MedImageForge is correctly configured                      |
| `cmd_explore()` | Scans the dataset, counts patients/images/masks, checks labels and demographics, and verifies file integrity | **Dataset validation** — tells us whether the dataset on disk matches what we expect |
| `cmd_inspect()` | Takes one patient's CT slice and creates a visualization of **bone + brain + segmentation mask**             | **Visual QC** — lets us manually verify that images and masks are correct            |
| `main()`        | Defines the CLI commands and decides which function to execute                                               | **Entry point/controller** — connects terminal commands to the correct functionality |


In [1]:
! python -m medimageforge info

medimageforge 0.1.0
log level: INFO

Configured paths:
  [OK     ] data_dir: /home/zahra/MedImageForge/data
  [OK     ] raw_dir: /home/zahra/MedImageForge/data/Patients_CT
  [OK     ] labels_csv: /home/zahra/MedImageForge/data/hemorrhage_diagnosis.csv
  [OK     ] demographics_csv: /home/zahra/MedImageForge/data/patient_demographics.csv
  [OK     ] checksums_file: /home/zahra/MedImageForge/data/SHA256SUMS.txt
  [OK     ] artifacts_dir: /home/zahra/MedImageForge/artifacts


In [2]:
! python -m medimageforge explore

=== Dataset report ===
Patients on disk:      82
Slices (brain):      2501
Slices (bone ):      2500
Segmentation masks:    318
Slices per patient:    min 38 / mean 61.0 / max 80

=== Labels (hemorrhage_diagnosis.csv) ===
Label rows:            2501
Patients with labels:  82
  Intraventricular     24
  Intraparenchymal     73
  Subarachnoid         18
  Epidural             173
  Subdural             56
  No_Hemorrhage        2183
  Fracture_Yes_No      195

=== Demographics (patient_demographics.csv) ===
Rows:                  82
Gender:                {'Male': np.int64(46), 'Female': np.int64(36)}

=== Labels vs files ===
Every label row has a matching brain slice.

=== Integrity (SHA256SUMS.txt) ===
2026-09-14 14:28:08,194 | INFO    | medimageforge.cli | Verifying checksums — hashing every file in data/ ...
Verified:   5325
Mismatched: 0
Missing:    0
Extra:      1
  - SHA256SUMS.txt

Integrity gate: PASS


In [5]:
! python -m medimageforge inspect 049 --slice 12

brain 12.jpg: {'shape': (650, 650), 'dtype': 'uint8', 'min': 0, 'max': 255, 'mean': 76.74, 'std': 96.77}
bone  12.jpg: {'shape': (650, 650), 'dtype': 'uint8', 'min': 0, 'max': 239, 'mean': 45.26, 'std': 46.79}
mask  12_HGE_Seg.jpg: none — overlay is plain brain window

Saved: /home/zahra/MedImageForge/artifacts/inspect/049_12.png
Panels: bone window | brain window | brain + mask overlay
